In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [11]:
# mappings baby
import string

chars = list(string.ascii_lowercase)
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [239]:
# dataset

block_size = 3 # context length: how many chars do we consider to predict next
X, Y = [], []
for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [49]:
# lookup table, embedding each of 27 characters into a 2d space
C = torch.randn((27, 2)) 
# C[X] X~(16, 3) C[X]~(16, 3, 2) since context window is 3. 
# indexing C basically = at first each ex in X is represented by 3 ints. now, each is represented by 3 pairs of floats

In [243]:
## initialize network
## X -> C[X] "embedding" -(W1, b1)-> H1 -(W2, b2)-> OL
# X~[B, 3], C[X]~[B, 3, 2], H1~[B, 32], OL~[B, 27]

g = torch.Generator().manual_seed(42)

C = torch.randn((27, 2), generator=g, requires_grad=True)
W1 = (torch.randn((block_size*2, 32), generator=g) * 0.01).requires_grad_()
b1 = (torch.randn((1, 32), generator=g) * 0.01).requires_grad_()
W2 = (torch.randn((32, 27), generator=g) * 0.01).requires_grad_()
b2 = (torch.randn((1, 27), generator=g) * 0.01).requires_grad_()

In [253]:
## training loop

for k in range(100):
    ## FORWARD PASS 
    # takes a batch of data and labels, computes loss based on current parameters
    embd = torch.flatten(C[X], start_dim=-2)
    H1 = (embd @ W1) + b1
    OL = (H1 @ W2) + b2
    logits = OL

    # softmax
    counts = logits.exp() # [B, 27]
    probs = counts / counts.sum(1, keepdims=True) # [B, 27]
    # probs[x, y] = probability of x being followed by y
    # here, x is an index to the 3 proceeding chars that make up the specific example in X
    # y is the next char - we only care about its probability because it actually does show up in the training data
    loss = -probs[torch.arange(X.shape[0]), Y].log().mean() + 0.1*(W1**2).mean()*(W2**2).mean()
    print(loss.item())

    ## BACKWARD PASS
    W1.grad = None
    b1.grad = None
    W2.grad = None
    b2.grad = None
    loss.backward()

    #update
    W1.data += -0.5 * W1.grad
    b1.data += -0.5 * b1.grad
    W2.data += -0.5 * W2.grad
    b2.data += -0.5 * b2.grad

2.711430311203003
2.7100813388824463
2.708775520324707
2.7075130939483643
2.7062928676605225
2.7051138877868652
2.7039754390716553
2.702876091003418
2.7018158435821533
2.700791835784912
2.6998040676116943
2.698850631713867
2.697929859161377
2.6970407962799072
2.6961817741394043
2.6953511238098145
2.694547653198242
2.693769931793213
2.693016290664673
2.6922855377197266
2.6915767192840576
2.690887689590454
2.690218687057495
2.6895675659179688
2.6889333724975586
2.6883153915405273
2.6877129077911377
2.687124729156494
2.6865501403808594
2.685988187789917
2.685438394546509
2.6849007606506348
2.6843738555908203
2.6838574409484863
2.6833505630493164
2.6828536987304688
2.682366132736206
2.681886911392212
2.6814162731170654
2.6809535026550293
2.6804988384246826
2.6800522804260254
2.679612398147583
2.6791799068450928
2.6787545680999756
2.678335666656494
2.6779236793518066
2.677518129348755
2.677119255065918
2.676726818084717
2.676340103149414
2.6759603023529053
2.675585985183716
2.67521810531616

In [263]:
# Sampling
g = torch.Generator().manual_seed(42)

for i in range(50):

    out = []
    ix = [0, 0, 0]
    while True:
        embd = torch.flatten(C[ix], start_dim=-2)
        H1 = (embd @ W1) + b1
        OL = (H1 @ W2) + b2
        logits = OL
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)

        ix[0] = ix[1]
        ix[1] = ix[2]
        ix[2] = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix[2]])
        if ix[2] == 0:
            break
    print(''.join(out))

anugeenvt.
a.
aabian.
danme.
an.
silayle.
aoemahlla.
an.
eaiacnnlenldsz.
.
ana.
mab.
edoe.
kalla.
mnqonynlah.
vahiaea.
aalamitnar.
tpheliia.
otcl.
ielaro.
dgan.
oar.
ukannern.
rmielsosri.
plyiewo.
liljahm.
fsi.
biadoyeln.
krok.
feiiavahia.
atoar.
bmrhl.
cdo.
tmdamaeaaea.
oet.
etiirvatla.
yaiarioeta.
e.
enaqnyiog.
acdan.
mkoriaya.
hlsoi.
i.
tn.
ahkaaniey.
cdenrn.
a.
kueheioaiia.
irn.
baiovrselaartol.
